# 03 - Evaluation controls and performance score

Is the low reliability a property of the data, or a bug in the chain? Controls:
median baseline, cross-validation, shuffled target, a **positive control** with real signal, seed stability.
Then the performance score (LLD 6.3 to 6.4) with thresholds and the effect of the budget.

In [1]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import common as c

c.OUT_DIR.mkdir(exist_ok=True)
(c.OUT_DIR / "figures").mkdir(exist_ok=True)
pd.set_option("display.precision", 4)
df = c.load_seed()
train, test = c.split(df)
spec = c.build_feature_spec(train)
x_train, x_test = c.encode(train, spec), c.encode(test, spec)
models = c.train_models(x_train, train)
predictions = c.predict(models, x_test)
metrics = c.evaluate(predictions, test, train)
summary: dict[str, object] = {}

## 1. Baselines: mean and median
A constant prediction is the bar to beat.

In [2]:
rows = {}
for t in c.TARGETS:
    y = test[t].to_numpy()
    rows[t] = {
        "model_mae": metrics[t]["mae"],
        "mean_baseline_mae": metrics[t]["baseline_mae"],
        "median_baseline_mae": float(np.mean(np.abs(y - train[t].median()))),
    }
baselines = pd.DataFrame(rows).T
display(baselines)
summary["baselines"] = rows

,model_mae,mean_baseline_mae,median_baseline_mae
click_through_rate,0.0733,0.0725,0.0726
conversion_rate,0.0485,0.0480,0.0480
website_visits,12.3977,12.3192,12.3140


## 2. 5-fold cross-validation on the training rows
Folds are cut with numpy (fixed seed). The test rows are not used. R2 stays around zero in every fold.

In [3]:
rng = np.random.default_rng(42)
folds = np.array_split(rng.permutation(len(train)), 5)
cv_rows = []
for k, held in enumerate(folds):
    fit_idx = np.concatenate([f for j, f in enumerate(folds) if j != k])
    fit, val = train.iloc[fit_idx], train.iloc[held]
    fold_models = c.train_models(c.encode(fit, spec), fit)
    fold_metrics = c.evaluate(c.predict(fold_models, c.encode(val, spec)), val, fit)
    cv_rows.append({"fold": k + 1, **{t: fold_metrics[t]["r2"] for t in c.TARGETS}})
cv = pd.DataFrame(cv_rows).set_index("fold")
display(cv)
print("mean R2 over folds:", cv.mean().round(4).to_dict())
summary["cv_mean_r2"] = cv.mean().to_dict()
assert (cv.mean() < 0.10).all()

,click_through_rate,conversion_rate,website_visits
fold,,,
1,-0.0402,-0.0190,-0.0348
2,-0.0321,-0.0409,-0.0510
3,-0.0408,-0.0375,-0.0230
4,-0.0426,-0.0326,-0.0528
5,-0.0416,-0.0286,-0.0628


mean R2 over folds: {'click_through_rate': -0.0395, 'conversion_rate': -0.0317, 'website_visits': -0.0449}


## 3. Shuffled-target control
Break the link between inputs and targets on purpose. R2 must not be positive: if it were, the evaluation would leak.

In [4]:
shuffled = train.copy()
for t in c.TARGETS:
    shuffled[t] = np.random.default_rng(1).permutation(shuffled[t].to_numpy())
shuffled_models = c.train_models(x_train, shuffled)
shuffled_metrics = c.evaluate(c.predict(shuffled_models, x_test), test, shuffled)
shuffled_r2 = {t: shuffled_metrics[t]["r2"] for t in c.TARGETS}
print("shuffled-target R2:", {k: round(v, 4) for k, v in shuffled_r2.items()})
summary["shuffled_r2"] = shuffled_r2
assert all(v < 0.02 for v in shuffled_r2.values())

shuffled-target R2: {'click_through_rate': -0.0305, 'conversion_rate': -0.0249, 'website_visits': -0.0228}


## 4. Positive control: data that really has a signal
Same rows and inputs, but the targets are rebuilt from the inputs plus noise (`common.strong_signal_copy`).
If the chain works, the models must find it: R2 >= 0.4 means reliability "good" (DATA_MODEL 4.7).

In [5]:
strong = c.strong_signal_copy(df)
s_train, s_test = c.split(strong)
s_models = c.train_models(c.encode(s_train, spec), s_train)
s_preds = c.predict(s_models, c.encode(s_test, spec))
s_metrics = c.evaluate(s_preds, s_test, s_train)
display(pd.DataFrame({t: s_metrics[t] for t in c.TARGETS}).T)
print("overall reliability with signal:", s_metrics["overall_reliability"])
summary["strong_signal_r2"] = {t: s_metrics[t]["r2"] for t in c.TARGETS}
for t in c.TARGETS:
    assert s_metrics[t]["r2"] >= 0.4, (t, s_metrics[t]["r2"])
assert s_metrics["overall_reliability"] == "good"

,mae,rmse,r2,baseline_mae,reliability
click_through_rate,0.0065,0.0082,0.9775,0.0449,good
conversion_rate,0.0063,0.008,0.9589,0.0326,good
website_visits,2.4482,3.0479,0.9036,8.1385,good


overall reliability with signal: good


## 5. Stability across seeds (information only)
Different `random_state` values change the result slightly; the frozen value stays 42 (LLD 5.4).

In [6]:
seed_rows = []
for seed in range(5):
    m = c.train_models(x_train, train, random_state=seed)
    e = c.evaluate(c.predict(m, x_test), test, train)
    seed_rows.append({"random_state": seed, **{t: e[t]["r2"] for t in c.TARGETS}})
seeds = pd.DataFrame(seed_rows).set_index("random_state")
display(seeds)
summary["seed_stability_r2_range"] = {t: [float(seeds[t].min()), float(seeds[t].max())] for t in c.TARGETS}

,click_through_rate,conversion_rate,website_visits
random_state,,,
0,-0.0391,-0.0439,-0.0295
1,-0.0412,-0.0425,-0.0340
2,-0.0409,-0.0408,-0.0352
3,-0.0351,-0.0391,-0.0241
4,-0.0390,-0.0368,-0.0243


## 6. Predicted vs actual and residuals (seed model vs signal model)

In [7]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for col, t in enumerate(c.TARGETS):
    axes[0, col].scatter(test[t], predictions[t], s=4, alpha=0.4, color="#5b7a99")
    axes[0, col].set_title(f"{t} - seed data")
    axes[0, col].set_xlabel("actual")
    axes[0, col].set_ylabel("predicted")
    axes[1, col].scatter(s_test[t], s_preds[t], s=4, alpha=0.4, color="#166534")
    axes[1, col].set_title(f"{t} - positive control")
    axes[1, col].set_xlabel("actual")
    axes[1, col].set_ylabel("predicted")
fig.tight_layout()
fig.savefig(c.OUT_DIR / "figures" / "03_predicted_vs_actual.png", dpi=110)
plt.show()

## 7. Performance score and thresholds (LLD 6.3, 6.4)
References are built from the real rows (training + test); thresholds are the terciles of the real scores.

In [8]:
reference = c.build_scoring_reference(df)
print(f"thresholds: low below {reference.low_below:.1f} | high from {reference.high_from:.1f}")
summary["thresholds"] = {"low_below": reference.low_below, "high_from": reference.high_from}
assert (round(reference.low_below, 1), round(reference.high_from, 1)) == (40.2, 60.8)

thresholds: low below 40.2 | high from 60.8


Score of every test row from the **predicted** figures (what a user would see), and from the real figures.

In [9]:
def scores_for(frame: pd.DataFrame, ctr, conv, visits) -> pd.Series:
    out = []
    for i, row in enumerate(frame.itertuples()):
        leads = c.estimated_leads(visits[i], conv[i])
        cpr = c.estimated_cost_per_result(row.ad_spend, leads)
        out.append(c.score_campaign(ctr[i], leads, cpr, row.campaign_type, reference.by_type, reference.all_types)["score"])
    return pd.Series(out, index=frame.index)

real_scores = scores_for(test, test["click_through_rate"].to_numpy(), test["conversion_rate"].to_numpy(), test["website_visits"].to_numpy())
seed_scores = scores_for(test, *(predictions[t] for t in c.TARGETS))
levels = seed_scores.map(lambda s: c.level_for(s, reference)).value_counts()
print("levels from seed-model predictions:", levels.to_dict())
print("rank correlation, predicted score vs real score (seed model):", round(seed_scores.corr(real_scores, method="spearman"), 3))

s_ref = c.build_scoring_reference(strong)
def strong_scores(frame, preds=None):
    out = []
    for i, row in enumerate(frame.itertuples()):
        a, b, v = (frame[t].to_numpy()[i] if preds is None else preds[t][i] for t in c.TARGETS)
        leads = c.estimated_leads(v, b)
        out.append(c.score_campaign(a, leads, c.estimated_cost_per_result(row.ad_spend, leads), row.campaign_type, s_ref.by_type, s_ref.all_types)["score"])
    return pd.Series(out, index=frame.index)

s_real = strong_scores(s_test)
s_pred = strong_scores(s_test, s_preds)
rank_corr = float(s_pred.corr(s_real, method="spearman"))
print("rank correlation, predicted score vs real score (positive control):", round(rank_corr, 3))
summary["score_rank_corr"] = {"seed": float(seed_scores.corr(real_scores, method="spearman")), "positive_control": rank_corr}
assert rank_corr > 0.5

levels from seed-model predictions: {'medium': 1135, 'high': 462, 'low': 5}
rank correlation, predicted score vs real score (seed model): 0.256
rank correlation, predicted score vs real score (positive control): 0.939


## 8. Score vs budget with constant (mean) predictions
With no signal, every campaign gets the same predicted rates, so the budget alone moves the cost note and the score.
This is why phase 12 recommendations can be dominated by the budget lever when the model has no signal (known limitation).

In [10]:
mean_ctr, mean_conv, mean_visits = (float(df[t].mean()) for t in c.TARGETS)
leads = c.estimated_leads(mean_visits, mean_conv)
rows = []
for budget in (500, 2500, 5000, 7500, 9500):
    result = c.score_campaign(mean_ctr, leads, c.estimated_cost_per_result(budget, leads), "Conversion", reference.by_type, reference.all_types)
    rows.append({"ad_spend": budget, "cost_note": round(result["cost_note"], 1), "score": result["score"], "level": c.level_for(result["score"], reference)})
budget_table = pd.DataFrame(rows).set_index("ad_spend")
display(budget_table)
summary["budget_table"] = rows

,cost_note,score,level
ad_spend,,,
500,95.6,69.0,high
2500,74.7,62.1,high
5000,54.6,55.4,medium
7500,42.9,51.5,medium
9500,35.8,49.1,medium


## 9. Save the summary

In [11]:
(c.OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2, default=float))
print("saved", c.OUT_DIR / "summary.json")

saved _out/summary.json


## Conclusion
* The low reliability on the seed is a property of the **data** (no signal): shuffled-target and cross-validation agree.
* The chain works: with real signal the same code reaches R2 above 0.9 and "good" reliability.
* The score and thresholds reproduce the values planned for phase 10 (about 40.2 and 60.8).
* Real agency data (imported in phase 5) is what will make the predictions useful.